# DPO & RLHF Alignment — From Scratch Masterclass

[![Python 3.10+](https://img.shields.io/badge/Python-3.10+-blue.svg)](https://www.python.org)
[![NumPy](https://img.shields.io/badge/NumPy-Pure%20Implementation-orange.svg)](https://numpy.org)
[![RLHF](https://img.shields.io/badge/Technique-RLHF%20%7C%20DPO-green.svg)]()

> **Zero black-boxes.** Every formula is derived, every component is implemented from scratch in pure Python + NumPy.
> This notebook is a companion to the portfolio's Production GenAI Systems and is targeted at MAANG L4–L5 interview preparation.

---

## 📖 What You Will Build

| Section | Technique | Key Outcome |
| :--- | :--- | :--- |
| **1** | Bradley-Terry Reward Model | Train a reward model on `(prompt, chosen, rejected)` pairs |
| **2** | PPO Conceptual Walkthrough | Understand why PPO is expensive and unstable |
| **3** | DPO Loss — Full Derivation | Implement the closed-form DPO objective from scratch |
| **4** | DPO Training Loop | Reference model (frozen) + policy model (trainable) |
| **5** | Evaluation | Win-rate, reward margins, β temperature sweep |

---

## 🔑 Prerequisites
```
numpy, matplotlib, scipy  — all standard; no GPU required
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import expit as sigmoid   # σ(x) = 1 / (1 + e^-x)
from scipy.stats import norm
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('dark_background')
ACCENT = '#00E5FF'
GREEN  = '#00FF88'
RED    = '#FF4B6E'
GOLD   = '#FFD700'

print('✅ Imports complete. NumPy:', np.__version__)

---
# Section 1 — Reward Modelling (RLHF Stage 1)

## 1.1 The Preference Learning Problem

RLHF starts with human annotators providing **pairwise preferences**:

> Given a prompt `x`, which response do you prefer: `y_w` (winner) or `y_l` (loser)?

The goal of **Stage 1 — Reward Modelling** is to learn a scalar reward function:

$$r_\phi(x, y) \rightarrow \mathbb{R}$$

that assigns higher reward to preferred (chosen) responses than to rejected ones.

## 1.2 Bradley-Terry Preference Model

The Bradley-Terry model says the probability that `y_w` is preferred over `y_l` given prompt `x` is:

$$P(y_w \succ y_l \mid x) = \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)$$

where $\sigma$ is the sigmoid function. The **reward model loss** is negative log-likelihood over the preference dataset $\mathcal{D}$:

$$\mathcal{L}_{\text{RM}}(\phi) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr) \right]$$

In [ ]:
# ── 1.3  Toy Preference Dataset ─────────────────────────────────────────────
# Each example is (prompt_features, chosen_features, rejected_features)
# Features are 8-dim hand-crafted vectors capturing:
#   [length_norm, factual_density, polite_tone, code_quality,
#    coherence, specificity, safety_score, helpfulness]

PREFERENCE_DATA: List[Dict] = [
    {"prompt": "Explain gradient descent",
     "chosen":   np.array([0.7, 0.9, 0.8, 0.2, 0.9, 0.8, 1.0, 0.9]),
     "rejected": np.array([0.3, 0.4, 0.5, 0.1, 0.5, 0.3, 1.0, 0.4])},
    {"prompt": "Write a Python quicksort",
     "chosen":   np.array([0.8, 0.7, 0.7, 0.9, 0.8, 0.9, 1.0, 0.8]),
     "rejected": np.array([0.5, 0.5, 0.6, 0.3, 0.5, 0.4, 1.0, 0.5])},
    {"prompt": "Summarise RLHF in 3 sentences",
     "chosen":   np.array([0.6, 0.8, 0.9, 0.1, 0.9, 0.7, 1.0, 0.9]),
     "rejected": np.array([0.9, 0.3, 0.4, 0.1, 0.4, 0.3, 0.9, 0.3])},
    {"prompt": "What is attention in transformers?",
     "chosen":   np.array([0.7, 0.9, 0.8, 0.3, 0.9, 0.9, 1.0, 0.9]),
     "rejected": np.array([0.4, 0.4, 0.5, 0.2, 0.5, 0.4, 1.0, 0.4])},
    {"prompt": "Explain LoRA fine-tuning",
     "chosen":   np.array([0.8, 0.8, 0.8, 0.6, 0.9, 0.8, 1.0, 0.9]),
     "rejected": np.array([0.3, 0.5, 0.6, 0.2, 0.5, 0.4, 1.0, 0.5])},
    {"prompt": "How does KV-caching work?",
     "chosen":   np.array([0.7, 0.9, 0.7, 0.4, 0.9, 0.8, 1.0, 0.8]),
     "rejected": np.array([0.5, 0.3, 0.5, 0.3, 0.4, 0.3, 1.0, 0.4])},
    {"prompt": "What is speculative decoding?",
     "chosen":   np.array([0.6, 0.8, 0.8, 0.3, 0.8, 0.7, 1.0, 0.8]),
     "rejected": np.array([0.4, 0.4, 0.6, 0.2, 0.4, 0.3, 1.0, 0.4])},
    {"prompt": "Compare BM25 vs dense retrieval",
     "chosen":   np.array([0.8, 0.9, 0.7, 0.2, 0.9, 0.9, 1.0, 0.9]),
     "rejected": np.array([0.4, 0.5, 0.5, 0.1, 0.5, 0.4, 1.0, 0.5])},
]

print(f'📊 Preference dataset: {len(PREFERENCE_DATA)} examples')
print(f'   Feature dim: {PREFERENCE_DATA[0]["chosen"].shape[0]}')

In [ ]:
# ── 1.4  Reward Model — Linear Scorer ────────────────────────────────────────
# r_φ(x, y) = φ · features(x, y)   (linear reward head, sufficient for toy dataset)

class RewardModel:
    """Linear Bradley-Terry reward model trained via gradient descent."""

    def __init__(self, feature_dim: int = 8, lr: float = 0.05):
        # φ — reward weight vector
        self.phi = np.random.randn(feature_dim) * 0.01
        self.lr  = lr
        self.loss_history: List[float] = []
        self.acc_history:  List[float] = []

    def score(self, features: np.ndarray) -> float:
        """r_φ(y) = φ · features"""
        return float(self.phi @ features)

    def preference_prob(self, chosen_feat: np.ndarray, rejected_feat: np.ndarray) -> float:
        """P(y_w ≻ y_l) = σ(r(y_w) - r(y_l))"""
        delta = self.score(chosen_feat) - self.score(rejected_feat)
        return float(sigmoid(delta))

    def compute_loss(self, dataset: List[Dict]) -> float:
        """L_RM = -E[log σ(r(y_w) - r(y_l))]"""
        losses = []
        for ex in dataset:
            prob = self.preference_prob(ex['chosen'], ex['rejected'])
            # Clip for numerical stability
            losses.append(-np.log(np.clip(prob, 1e-7, 1.0)))
        return float(np.mean(losses))

    def train_step(self, dataset: List[Dict]) -> Tuple[float, float]:
        """One epoch of gradient descent via analytic gradient of L_RM w.r.t. φ."""
        grad = np.zeros_like(self.phi)
        correct = 0
        for ex in dataset:
            p = self.preference_prob(ex['chosen'], ex['rejected'])
            # ∂L/∂φ = -(1 - p) * (chosen_feat - rejected_feat)
            grad += -(1.0 - p) * (ex['chosen'] - ex['rejected'])
            if p > 0.5:
                correct += 1
        grad /= len(dataset)
        self.phi -= self.lr * grad
        loss = self.compute_loss(dataset)
        acc  = correct / len(dataset)
        self.loss_history.append(loss)
        self.acc_history.append(acc)
        return loss, acc

    def train(self, dataset: List[Dict], epochs: int = 200) -> None:
        for epoch in range(epochs):
            loss, acc = self.train_step(dataset)
            if (epoch + 1) % 50 == 0:
                print(f'  Epoch {epoch+1:>4d} | Loss: {loss:.4f} | Pref-Acc: {acc*100:.1f}%')


print('Training Reward Model (Bradley-Terry)...')
rm = RewardModel(feature_dim=8, lr=0.08)
rm.train(PREFERENCE_DATA, epochs=300)

In [ ]:
# ── 1.5  Visualise Reward Model Training ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Section 1 — Reward Model Training (Bradley-Terry)', fontsize=14, color=ACCENT)

# Loss curve
axes[0].plot(rm.loss_history, color=RED, lw=2)
axes[0].set_title('Training Loss  L_RM', color='white')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Negative Log-Likelihood')
axes[0].grid(alpha=0.2)

# Preference accuracy
axes[1].plot(rm.acc_history, color=GREEN, lw=2)
axes[1].axhline(1.0, color='white', ls='--', lw=1, alpha=0.4, label='Perfect')
axes[1].set_title('Preference Accuracy', color='white')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Fraction Correct')
axes[1].legend(); axes[1].grid(alpha=0.2)

# Reward score distributions
chosen_scores   = [rm.score(ex['chosen'])   for ex in PREFERENCE_DATA]
rejected_scores = [rm.score(ex['rejected']) for ex in PREFERENCE_DATA]
axes[2].scatter(range(len(chosen_scores)),   chosen_scores,   color=GREEN, s=100, label='Chosen (y_w)',   zorder=5)
axes[2].scatter(range(len(rejected_scores)), rejected_scores, color=RED,   s=100, label='Rejected (y_l)', zorder=5, marker='x')
for i in range(len(PREFERENCE_DATA)):
    axes[2].plot([i, i], [rejected_scores[i], chosen_scores[i]], color='gray', alpha=0.4, lw=1.5)
axes[2].axhline(0, color='white', ls='--', lw=1, alpha=0.3)
axes[2].set_title('Reward Scores per Preference Pair', color='white')
axes[2].set_xlabel('Example Index'); axes[2].set_ylabel('r_φ(y)')
axes[2].legend(); axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

margin = np.mean([c - r for c, r in zip(chosen_scores, rejected_scores)])
print(f'\n📊 Mean reward margin (chosen - rejected): {margin:.4f}')
print(f'   Final preference accuracy: {rm.acc_history[-1]*100:.1f}%')

---
# Section 2 — PPO: Why It's Powerful But Expensive

## 2.1 RLHF Stage 2: Reinforcement Learning

With a trained reward model $r_\phi$, we now want to tune the LLM policy $\pi_\theta$ to maximise expected reward:

$$\mathcal{J}(\theta) = \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta(\cdot|x)} \bigl[ r_\phi(x, y) \bigr] - \beta \cdot \mathbb{KL}\bigl[\pi_\theta(\cdot|x) \| \pi_{\text{ref}}(\cdot|x)\bigr]$$

The **KL penalty** with weight $\beta$ prevents the policy from deviating catastrophically from the reference SFT model.

## 2.2 The PPO Update Rule

PPO clips the policy ratio $r_t = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ to stabilise training:

$$\mathcal{L}_{\text{PPO}} = -\mathbb{E}_t \left[ \min\bigl(r_t \hat{A}_t,\ \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon)\hat{A}_t\bigr) \right]$$

## 2.3 Why PPO is a Pain in Production

| Problem | Detail |
| :--- | :--- |
| **4 models in memory** | Policy, Reference, Reward Model, Value/Critic — all GPU-resident |
| **Online sampling** | Must generate from the policy *during training* → massive GPU-hours |
| **Reward hacking** | Policy finds degenerate responses that fool the reward model |
| **Hyperparameter sensitivity** | ε, β, GAE-λ, clip ratio — all interact |
| **Training instability** | Value function collapse, entropy collapse |

→ This motivated **DPO: Direct Preference Optimisation** (Rafailov et al., 2023)

In [ ]:
# ── 2.4  KL Divergence — Live Demonstration ──────────────────────────────────
# KL[π_θ || π_ref] = Σ π_θ(y) log(π_θ(y) / π_ref(y))

def kl_divergence(p: np.ndarray, q: np.ndarray, eps: float = 1e-10) -> float:
    """KL(P || Q) — asymmetric divergence from Q to P."""
    p = np.clip(p, eps, 1.0); p /= p.sum()
    q = np.clip(q, eps, 1.0); q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

# Simulate policy drift from reference
drift_strengths = np.linspace(0, 3.0, 50)
kl_values = []
vocab_size = 50

ref_logits = np.random.randn(vocab_size)
ref_probs  = np.exp(ref_logits) / np.exp(ref_logits).sum()

for drift in drift_strengths:
    policy_logits = ref_logits + drift * np.random.randn(vocab_size)
    policy_probs  = np.exp(policy_logits) / np.exp(policy_logits).sum()
    kl_values.append(kl_divergence(policy_probs, ref_probs))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(drift_strengths, kl_values, color=ACCENT, lw=2.5)
ax.fill_between(drift_strengths, kl_values, alpha=0.15, color=ACCENT)
ax.axvline(1.0, color=GOLD, ls='--', lw=1.5, label='Acceptable drift')
ax.set_title('KL Divergence vs Policy Drift Strength\n(the cost of moving away from π_ref)', color='white', fontsize=13)
ax.set_xlabel('Drift Strength'); ax.set_ylabel('KL[π_θ || π_ref]')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print('KL divergence grows super-linearly with drift — the β coefficient controls this trade-off.')

---
# Section 3 — Direct Preference Optimisation (DPO) — Full Derivation

## 3.1 Key Insight: Reward as a Function of Log-Ratios

Rafailov et al. (2023) showed that the optimal policy under the KL-constrained RLHF objective has a closed-form:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\!\left(\frac{1}{\beta} r(x, y)\right)$$

Rearranging, we can express the reward **as a function of policy log-ratios**:

$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

## 3.2 The DPO Loss

Substituting into the Bradley-Terry preference probability and noting that $Z(x)$ cancels:

$$\boxed{\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma\!\left( \beta \underbrace{\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}}_{\text{chosen advantage}} - \beta \underbrace{\log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}}_{\text{rejected advantage}} \right) \right]}$$

## 3.3 Why DPO is Elegant

| PPO | DPO |
| :--- | :--- |
| Needs explicit reward model | Reward model is *implicit* in the loss |
| Online sampling during training | Offline — uses fixed preference dataset |
| 4 models in GPU memory | 2 models only: policy + reference |
| Hyperparameter-heavy | Just β to tune |
| Used in: InstructGPT, early ChatGPT | Used in: Llama-3, Mistral-Instruct, Gemma |

In [ ]:
# ── 3.4  DPO Loss — Implementation ───────────────────────────────────────────

def log_policy_ratio(
    policy_logprob_chosen:   float,
    ref_logprob_chosen:      float,
    policy_logprob_rejected: float,
    ref_logprob_rejected:    float,
    beta:                    float = 0.1,
) -> float:
    """Compute the DPO margin inside σ(·).

    margin = β * [log π_θ(y_w|x) - log π_ref(y_w|x)]
           - β * [log π_θ(y_l|x) - log π_ref(y_l|x)]
    """
    chosen_advantage   = policy_logprob_chosen   - ref_logprob_chosen
    rejected_advantage = policy_logprob_rejected - ref_logprob_rejected
    return beta * (chosen_advantage - rejected_advantage)


def dpo_loss(
    policy_logprobs_chosen:   np.ndarray,
    ref_logprobs_chosen:      np.ndarray,
    policy_logprobs_rejected: np.ndarray,
    ref_logprobs_rejected:    np.ndarray,
    beta:                     float = 0.1,
) -> float:
    """Batch DPO loss over N preference examples.

    L_DPO = -mean( log σ( β*(log π_θ(y_w) - log π_ref(y_w))
                        - β*(log π_θ(y_l) - log π_ref(y_l)) ) )
    """
    margins = beta * (
        (policy_logprobs_chosen   - ref_logprobs_chosen) -
        (policy_logprobs_rejected - ref_logprobs_rejected)
    )
    return float(-np.mean(np.log(sigmoid(margins))))


def preference_accuracy(
    policy_logprobs_chosen:   np.ndarray,
    ref_logprobs_chosen:      np.ndarray,
    policy_logprobs_rejected: np.ndarray,
    ref_logprobs_rejected:    np.ndarray,
    beta:                     float = 0.1,
) -> float:
    """Fraction of examples where DPO assigns higher probability to chosen vs rejected."""
    margins = beta * (
        (policy_logprobs_chosen   - ref_logprobs_chosen) -
        (policy_logprobs_rejected - ref_logprobs_rejected)
    )
    return float(np.mean(margins > 0))


print('✅ DPO loss functions defined.')

# Quick sanity check
demo_margin = log_policy_ratio(
    policy_logprob_chosen=-1.2, ref_logprob_chosen=-1.5,
    policy_logprob_rejected=-2.1, ref_logprob_rejected=-1.8,
    beta=0.1
)
print(f'\nSanity check — DPO margin: {demo_margin:.4f}')
print(f'  σ(margin) = {sigmoid(demo_margin):.4f}  (>0.5 means chosen is preferred ✅)')

---
# Section 4 — DPO Training Loop From Scratch

We implement a minimal DPO trainer with:
- A **reference model** $\pi_{\text{ref}}$ (frozen SFT checkpoint — simulated as fixed log-prob vectors)
- A **policy model** $\pi_\theta$ (trainable — we learn a score vector θ)
- A **DPO training loop** updating θ to minimise $\mathcal{L}_{\text{DPO}}$

In [ ]:
# ── 4.1  Simulated Log-Probability Model ─────────────────────────────────────
# In production: log π(y|x) = sum of per-token log-probs from the LLM.
# Here: log π_θ(y|x) = θ · features(y)  (parametric approximation).

class SimplePolicyModel:
    """Toy parametric policy that produces log-probabilities via a learned weight vector.

    log π_θ(y|x) ≈ θ · features(y) + noise
    """

    def __init__(self, feature_dim: int = 8, frozen: bool = False, seed: int = 0):
        rng = np.random.RandomState(seed)
        self.theta  = rng.randn(feature_dim) * 0.5
        self.frozen = frozen
        self.feature_dim = feature_dim

    def log_prob(self, features: np.ndarray, noise_scale: float = 0.05) -> float:
        """log π_θ(y|x) — normalised to be a valid log-probability (< 0)."""
        raw = float(self.theta @ features)
        # Map to negative log-prob space: larger dot product → less negative log-prob
        return -np.log1p(np.exp(-raw + np.random.randn() * noise_scale))

    def compute_logprobs(
        self,
        dataset: List[Dict],
        key: str,
        noise_scale: float = 0.02,
    ) -> np.ndarray:
        """Compute log-probs for all examples in the dataset for a given response key."""
        return np.array([self.log_prob(ex[key], noise_scale) for ex in dataset])


# Reference model — frozen SFT checkpoint
ref_model    = SimplePolicyModel(feature_dim=8, frozen=True, seed=7)
# Policy model — starts from same init as ref (standard practice)
policy_model = SimplePolicyModel(feature_dim=8, frozen=False, seed=7)

print('Reference model θ (frozen):', np.round(ref_model.theta, 3))
print('Policy model θ (init):     ', np.round(policy_model.theta, 3))
print('\n✅ Both models initialised from the same weights (standard RLHF practice).')

In [ ]:
# ── 4.2  DPO Trainer ─────────────────────────────────────────────────────────

class DPOTrainer:
    """Minimal DPO training loop.

    Updates the policy model θ to minimise:
      L_DPO(θ) = -E[ log σ( β*(log π_θ(y_w) - log π_ref(y_w))
                           - β*(log π_θ(y_l) - log π_ref(y_l)) ) ]
    """

    def __init__(
        self,
        policy:    SimplePolicyModel,
        reference: SimplePolicyModel,
        beta:      float = 0.1,
        lr:        float = 0.03,
    ):
        self.policy    = policy
        self.reference = reference
        self.beta      = beta
        self.lr        = lr
        self.loss_history:    List[float] = []
        self.acc_history:     List[float] = []
        self.margin_history:  List[float] = []

    def _compute_margins(self, dataset: List[Dict]) -> np.ndarray:
        """β * [(log π_θ(y_w) - log π_ref(y_w)) - (log π_θ(y_l) - log π_ref(y_l))]"""
        pi_chosen   = self.policy.compute_logprobs(dataset, 'chosen')
        ref_chosen  = self.reference.compute_logprobs(dataset, 'chosen')
        pi_rejected = self.policy.compute_logprobs(dataset, 'rejected')
        ref_rejected= self.reference.compute_logprobs(dataset, 'rejected')
        return self.beta * ((pi_chosen - ref_chosen) - (pi_rejected - ref_rejected))

    def train_step(self, dataset: List[Dict]) -> Tuple[float, float]:
        """One gradient step on the DPO objective."""
        margins = self._compute_margins(dataset)
        probs   = sigmoid(margins)                  # σ(margin)
        loss    = float(-np.mean(np.log(np.clip(probs, 1e-7, 1.0))))
        acc     = float(np.mean(margins > 0))

        # ∂L_DPO/∂θ via chain rule (analytical gradient for linear policy)
        # dL/dmargin = -(1 - σ(margin)) / N  (from d/dm [-log σ(m)] = -(1-σ(m)))
        # dmargin/dθ ≈ β * (chosen_feat - rejected_feat)  (linear policy approx)
        grad = np.zeros_like(self.policy.theta)
        for i, ex in enumerate(dataset):
            dl_dm = -(1.0 - probs[i])
            dm_dtheta = self.beta * (ex['chosen'] - ex['rejected'])
            grad += dl_dm * dm_dtheta
        grad /= len(dataset)
        self.policy.theta -= self.lr * grad

        self.loss_history.append(loss)
        self.acc_history.append(acc)
        self.margin_history.append(float(np.mean(margins)))
        return loss, acc

    def train(self, dataset: List[Dict], epochs: int = 300) -> None:
        print(f'Training DPO policy (β={self.beta}, lr={self.lr})...')
        for epoch in range(epochs):
            loss, acc = self.train_step(dataset)
            if (epoch + 1) % 75 == 0:
                print(f'  Epoch {epoch+1:>4d} | L_DPO: {loss:.4f} | Pref-Acc: {acc*100:.1f}%')


trainer = DPOTrainer(policy_model, ref_model, beta=0.1, lr=0.04)
trainer.train(PREFERENCE_DATA, epochs=400)

In [ ]:
# ── 4.3  DPO Training Visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Section 4 — DPO Training Loop', fontsize=14, color=ACCENT)

axes[0].plot(trainer.loss_history, color=RED, lw=2)
axes[0].set_title('DPO Loss  L_DPO(θ)', color='white')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.2)

axes[1].plot(trainer.acc_history, color=GREEN, lw=2)
axes[1].axhline(1.0, color='white', ls='--', lw=1, alpha=0.4)
axes[1].set_title('Preference Accuracy', color='white')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Fraction Correct')
axes[1].grid(alpha=0.2)

axes[2].plot(trainer.margin_history, color=GOLD, lw=2)
axes[2].axhline(0, color='white', ls='--', lw=1, alpha=0.4)
axes[2].set_title('Mean DPO Margin (chosen advantage)', color='white')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('β * (chosen_adv - rejected_adv)')
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

---
# Section 5 — Evaluation: Win-Rate, Reward Margins & β Sweep

## 5.1 Evaluation Metrics

| Metric | Formula | Interpretation |
| :--- | :--- | :--- |
| **Win-Rate** | `P(π_θ(y_w) > π_θ(y_l))` | Does policy prefer chosen over rejected? |
| **Reward Margin** | `r(y_w) - r(y_l)` scored by reward model | Reward model's view of alignment |
| **KL from Reference** | `KL[π_θ || π_ref]` | How far did we drift from SFT? |

## 5.2 β Temperature Sweep

- **Low β (< 0.05)**: Policy ignores KL penalty → potential reward hacking, over-optimisation
- **β = 0.1**: Standard starting point (Rafailov et al. recommendation)
- **High β (> 0.5)**: Policy stays close to reference → under-optimised, conservative

In [ ]:
# ── 5.3  β Temperature Sweep ─────────────────────────────────────────────────
beta_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
sweep_results = []

for beta in beta_values:
    # Fresh policy model for each β
    sweep_policy = SimplePolicyModel(feature_dim=8, frozen=False, seed=7)
    sweep_trainer = DPOTrainer(sweep_policy, ref_model, beta=beta, lr=0.04)
    
    # Train silently
    for _ in range(400):
        sweep_trainer.train_step(PREFERENCE_DATA)
    
    final_loss = sweep_trainer.loss_history[-1]
    final_acc  = sweep_trainer.acc_history[-1]
    final_margin = sweep_trainer.margin_history[-1]
    
    # Compute policy drift (proxy KL via L2 distance in θ-space)
    policy_drift = float(np.linalg.norm(sweep_policy.theta - ref_model.theta))
    
    sweep_results.append({
        'beta': beta,
        'loss': final_loss,
        'acc': final_acc,
        'margin': final_margin,
        'drift': policy_drift,
    })
    print(f'β={beta:<5} | Loss={final_loss:.4f} | Acc={final_acc*100:.1f}% | Margin={final_margin:.4f} | Drift={policy_drift:.4f}')

print('\n✅ β sweep complete.')

In [ ]:
# ── 5.4  Visualise β Sweep ───────────────────────────────────────────────────
betas    = [r['beta']  for r in sweep_results]
accs     = [r['acc']   for r in sweep_results]
margins  = [r['margin']for r in sweep_results]
drifts   = [r['drift'] for r in sweep_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Section 5 — β Temperature Sweep (DPO Alignment vs Conservatism)', fontsize=13, color=ACCENT)

axes[0].semilogx(betas, [a*100 for a in accs], 'o-', color=GREEN, lw=2, ms=8)
axes[0].axvline(0.1, color=GOLD, ls='--', lw=1.5, label='β=0.1 (recommended)')
axes[0].set_title('Preference Accuracy vs β', color='white')
axes[0].set_xlabel('β (log scale)'); axes[0].set_ylabel('Win-Rate %')
axes[0].legend(); axes[0].grid(alpha=0.2)

axes[1].semilogx(betas, margins, 'o-', color=ACCENT, lw=2, ms=8)
axes[1].axhline(0, color='white', ls='--', lw=1, alpha=0.4)
axes[1].axvline(0.1, color=GOLD, ls='--', lw=1.5)
axes[1].set_title('Mean DPO Margin vs β', color='white')
axes[1].set_xlabel('β (log scale)'); axes[1].set_ylabel('Margin')
axes[1].grid(alpha=0.2)

axes[2].semilogx(betas, drifts, 'o-', color=RED, lw=2, ms=8)
axes[2].axvline(0.1, color=GOLD, ls='--', lw=1.5, label='β=0.1')
axes[2].set_title('Policy Drift from π_ref vs β', color='white')
axes[2].set_xlabel('β (log scale)'); axes[2].set_ylabel('||θ_policy - θ_ref||₂')
axes[2].legend(); axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print('\n📊 Key Takeaways:')
print('  • Low β  → High margin + high drift → Over-optimisation / reward hacking risk')
print('  • High β → Low drift (conservative) → Under-optimisation')
print('  • β=0.1 balances alignment and conservatism ✅')

In [ ]:
# ── 5.5  Win-Rate Comparison: Reference vs DPO Policy ────────────────────────
def compute_win_rate(
    policy: SimplePolicyModel,
    reference: SimplePolicyModel,
    dataset: List[Dict],
) -> Dict:
    """How often does the DPO policy prefer chosen over rejected vs reference model?"""
    policy_wins    = 0
    reference_wins = 0
    for ex in dataset:
        pi_chosen  = policy.log_prob(ex['chosen'])
        pi_reject  = policy.log_prob(ex['rejected'])
        ref_chosen = reference.log_prob(ex['chosen'])
        ref_reject = reference.log_prob(ex['rejected'])
        if pi_chosen > pi_reject:    policy_wins    += 1
        if ref_chosen > ref_reject:  reference_wins += 1
    n = len(dataset)
    return {
        'dpo_policy_win_rate':  policy_wins    / n,
        'reference_win_rate':   reference_wins / n,
    }


win_rates = compute_win_rate(policy_model, ref_model, PREFERENCE_DATA)

fig, ax = plt.subplots(figsize=(8, 5))
models  = ['Reference Model\n(π_ref, frozen SFT)', 'DPO Policy\n(π_θ after training)']
rates   = [win_rates['reference_win_rate'] * 100, win_rates['dpo_policy_win_rate'] * 100]
colors  = [RED, GREEN]
bars = ax.bar(models, rates, color=colors, width=0.4, edgecolor='white', linewidth=0.8)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{rate:.1f}%', ha='center', va='bottom', color='white', fontweight='bold', fontsize=13)
ax.set_ylim(0, 115)
ax.set_title('Win-Rate: Prefers Chosen over Rejected', color='white', fontsize=13)
ax.set_ylabel('Win-Rate (%)')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

print(f'\n🏆 DPO Policy Win-Rate:  {win_rates["dpo_policy_win_rate"]*100:.1f}%')
print(f'   Reference Win-Rate:   {win_rates["reference_win_rate"]*100:.1f}%')

---
# 📊 Summary & Interview-Ready Takeaways

## What We Built End-to-End

```
Preference Data (x, y_w, y_l)
         │
         ▼
  ┌──────────────────────────────┐
  │  Section 1: Reward Model     │  Bradley-Terry, L_RM = -E[log σ(r(y_w) - r(y_l))]
  └──────────────────────────────┘
         │
         ▼
  ┌──────────────────────────────┐
  │  Section 2: PPO              │  Expensive: 4 models, online sampling, KL penalty
  └──────────────────────────────┘
         │
         ▼ (motivation for DPO)
  ┌──────────────────────────────┐
  │  Section 3: DPO Loss         │  L_DPO = -E[log σ(β*(log π_θ/π_ref(y_w) - log π_θ/π_ref(y_l)))]
  └──────────────────────────────┘
         │
         ▼
  ┌──────────────────────────────┐
  │  Section 4: DPO Training     │  Reference (frozen) + Policy (trainable), gradient descent
  └──────────────────────────────┘
         │
         ▼
  ┌──────────────────────────────┐
  │  Section 5: Evaluation       │  Win-rate, reward margin, β sweep
  └──────────────────────────────┘
```

## Key Interview Answers

**Q: What's the difference between RLHF and DPO?**
> RLHF requires training an explicit reward model then running PPO online — 4 models in memory, expensive. DPO derives the same optimal policy from a closed-form reparametrisation: the reward model is implicit in the log-ratio of policy to reference. Training is offline on a static preference dataset. 2 models, no sampling.

**Q: What does β control in DPO?**
> β is the KL penalty coefficient. Low β → policy drifts far from reference (potential reward hacking). High β → policy stays conservative (under-optimised). β=0.1 is the typical starting point per Rafailov et al.

**Q: When would you still use PPO over DPO?**
> When you need online exploration, e.g., multi-step reasoning, tool use, code execution tasks where offline preference data isn't sufficient. PPO can also handle reward signals from environments (not just human prefs).

## Further Reading
- Rafailov et al. (2023) — *Direct Preference Optimization: Your Language Model is Secretly a Reward Model* — https://arxiv.org/abs/2305.18290
- Schulman et al. (2017) — *Proximal Policy Optimization* — https://arxiv.org/abs/1707.06347
- Ouyang et al. (2022) — *Training language models to follow instructions with human feedback* (InstructGPT) — https://arxiv.org/abs/2203.02155